In [ ]:
# [1/6] Mount Google Drive & Setup Path
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/code')

print('Project root: /content/drive/MyDrive/code')


In [ ]:
# [2/6] Install Dependencies
!pip install -q transformers peft accelerate bitsandbytes "torchao>=0.16.0" 2>/dev/null

import sys
sys.path.insert(0, '/content/drive/MyDrive/code')

print('Ready.')


In [ ]:
# [3/6] CONFIG — change these

# Model tag (fixed to Ministral8b — this notebook is the Ministral selector variant)
MODEL_NAME = "Ministral8b"

# Dataset used to generate selector data: "val" | "test" | "test_folio" | "test_willow"
DATASET = "val"

# Training hyperparams
EPOCHS = 6
BATCH_SIZE = 4
GRAD_ACCUM = 2
LR = 1e-4
MAX_LENGTH = 512
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 1

# LoRA hyperparams
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

print(f"Model:       {MODEL_NAME}")
print(f"Data:        {MODEL_NAME}_k10_{DATASET}_selector_*.json")
print(f"Selector:    m4 (BCE + sigmoid)")
print(f"Epochs:      {EPOCHS}")
print(f"Batch size:  {BATCH_SIZE} x {GRAD_ACCUM}")
print(f"LR:          {LR}")
print(f"Early stop:  patience={EARLY_STOPPING_PATIENCE}")


In [ ]:
# [4/6] Load selector data
#
# Prereq: notebooks/selection/selector_data.ipynb with MODEL = "Ministral8b"
# Reads:  data/results/Ministral8b/k10/Ministral8b_k10_{DATASET}_selector_{train,val}.json

import json
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader

_ROOT = Path('/content/drive/MyDrive/code')
RESULTS_DIR = _ROOT / 'data' / 'results' / MODEL_NAME / 'k10'

_suffix = DATASET.replace('test_', '').replace('test', '')
_tag = f"{MODEL_NAME}_k10_{_suffix}" if _suffix else f"{MODEL_NAME}_k10"
train_path = RESULTS_DIR / f"{_tag}_selector_train.json"
val_path = RESULTS_DIR / f"{_tag}_selector_val.json"

for p in (train_path, val_path):
    if not p.exists():
        raise FileNotFoundError(
            f"Selector data not found: {p}\n"
            f"Run notebooks/selection/selector_data.ipynb (MODEL='{MODEL_NAME}') first."
        )

with open(train_path, encoding='utf-8') as f:
    train_samples = json.load(f)['samples']
with open(val_path, encoding='utf-8') as f:
    val_samples = json.load(f)['samples']

print(f"Train samples: {len(train_samples)}")
print(f"Val samples:   {len(val_samples)}")


class FOLSelectorDataset(Dataset):
    """(NL, FOL) -> label dataset for BCE training."""

    def __init__(self, samples, tokenizer, max_length=512):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        text = f"NL: {s['nl']}\nFOL: {s['fol']}"
        enc = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(float(s['label']), dtype=torch.float),
        }


In [ ]:
# [5/6] Load tokenizer + Ministral selector model (Mistral3 + LoRA + score_head)

import os
import torch.nn as nn
from transformers import Mistral3ForConditionalGeneration, AutoTokenizer
from peft import LoraConfig, get_peft_model

os.environ.setdefault('HF_HUB_DISABLE_IMPLICIT_TOKEN', '1')

BASE_MODEL = 'mistralai/Ministral-3-8B-Instruct-2512'

# ---- tokenizer (fix_mistral_regex for the broken Mistral tokenizer regex) ----
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, trust_remote_code=True, fix_mistral_regex=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


class MinistralSelector(nn.Module):
    """Mistral3 text-backbone + LoRA + score_head -> sigmoid (BCE)."""

    def __init__(self, base_model_name, lora_config):
        super().__init__()
        # Ministral-3-8B-Instruct-2512 is an FP8 pre-quantized checkpoint on the
        # Hub (quant_method="fp8"), so BitsAndBytes 4-bit re-quantization is
        # rejected by transformers. Load WITHOUT a quantization_config: the FP8
        # weights stay FP8 and bf16 is used for the non-quantized modules
        # (vision_tower / multi_modal_projector / lm_head). Lower BATCH_SIZE if OOM.
        base = Mistral3ForConditionalGeneration.from_pretrained(
            base_model_name, torch_dtype=torch.bfloat16,
            device_map='auto', trust_remote_code=True,
        )
        base.config.use_cache = False
        base.gradient_checkpointing_enable()
        self.llm = get_peft_model(base, lora_config)
        hidden_size = base.config.text_config.hidden_size  # 4096
        self.score_head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        # Mistral3 is a VLM. Reach the text backbone directly to skip the
        # lm_head (vocab=131072) and read last_hidden_state cheaply:
        #   self.llm (Peft) -> .base_model.model (ForCondGen)
        #                    -> .model (Mistral3Model) -> .language_model
        text_model = self.llm.base_model.model.model.language_model
        outputs = text_model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state[:, -1, :].float()  # last token
        logit = self.score_head(last_hidden).squeeze(-1)
        return torch.sigmoid(logit)


# ---- LoRA config (same targets as the Qwen selector / Ministral formalizer) ----
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none', task_type='CAUSAL_LM',
)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'  VRAM: {vram_gb:.1f} GB  (FP8 checkpoint, bf16 compute)')

print('  Loading model + LoRA...')
model = MinistralSelector(BASE_MODEL, lora_config)
device = next(model.llm.parameters()).device
model.to(device)

# ---- datasets / loaders ----
train_ds = FOLSelectorDataset(train_samples, tokenizer, max_length=MAX_LENGTH)
val_ds = FOLSelectorDataset(val_samples, tokenizer, max_length=MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Device:           {device}")
print(f"  Trainable params: {n_trainable:,}")

# ---- smoke test: one forward pass to verify the Mistral3 text-backbone path ----
sample = next(iter(train_loader))
with torch.no_grad():
    preds = model(input_ids=sample['input_ids'].to(device),
                  attention_mask=sample['attention_mask'].to(device))
print(f"  Smoke test OK — preds {tuple(preds.shape)}, "
      f"range [{preds.min().item():.3f}, {preds.max().item():.3f}]")


In [ ]:
# [6/6] Train M4 selector (BCE + sigmoid -> continuous [0,1])
#
# Saves: models/Ministral8b_selector/  (LoRA adapter + score_head.pt + tokenizer)
# Early-stops when val_loss fails to improve for EARLY_STOPPING_PATIENCE+1 epochs.

import numpy as np
from datetime import datetime
from transformers import get_linear_schedule_with_warmup

MODEL_DIR = _ROOT / 'models'
out_dir = MODEL_DIR / f"{MODEL_NAME}_selector"
out_dir.mkdir(parents=True, exist_ok=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
)

bce = nn.BCELoss()


def train_epoch(accum_steps=GRAD_ACCUM):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        preds = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = bce(preds, labels) / accum_steps
        loss.backward()

        if (batch_idx + 1) % accum_steps == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps
        if batch_idx % 200 == 0:
            print(f"    batch {batch_idx:>4}/{len(train_loader)}  "
                  f"loss={loss.item() * accum_steps:.4f}")

    return total_loss / len(train_loader)


@torch.no_grad()
def eval_epoch():
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        preds = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = bce(preds, labels)
        total_loss += loss.item()
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    acc = np.mean((np.array(all_preds) >= 0.5) == np.array(all_labels))
    return total_loss / len(val_loader), float(acc)


# ---- resume from checkpoint ----
ckpt_path = out_dir / 'checkpoint.pt'
start_epoch = 0
best_val_loss = float('inf')
patience_counter = 0

if ckpt_path.exists():
    print('  Checkpoint found — resuming...')
    ckpt = torch.load(str(ckpt_path), map_location=device)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt['best_val_loss']
    patience_counter = ckpt.get('patience_counter', 0)
    print(f"  Resumed from epoch {start_epoch + 1}  |  "
          f"best_val_loss={best_val_loss:.4f}  patience={patience_counter}")
    print()

print('=' * 55)
print(f"  M4 SELECTOR FINE-TUNING  ({MODEL_NAME} + LoRA + head)")
print(f"  Data:    {_tag}_selector_*.json")
print(f"  Train:   {len(train_samples)}  Val: {len(val_samples)}")
print(f"  Epochs:  {EPOCHS}  |  Batch: {BATCH_SIZE} x {GRAD_ACCUM}  |  LR: {LR}")
print(f"  Output:  {out_dir}")
print('=' * 55)
print()

t0 = datetime.now()
stopped_early = False

for epoch in range(start_epoch, EPOCHS):
    print(f"  Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_epoch()
    val_loss, val_acc = eval_epoch()
    print(f"    train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save LoRA adapter + score_head + tokenizer
        model.llm.save_pretrained(str(out_dir))
        torch.save(model.score_head.state_dict(), str(out_dir / 'score_head.pt'))
        tokenizer.save_pretrained(str(out_dir))
        print('    -> saved (best)')
    else:
        patience_counter += 1
        print(f"    -> no improvement (patience {patience_counter}/{EARLY_STOPPING_PATIENCE + 1})")
        if patience_counter > EARLY_STOPPING_PATIENCE:
            print(f"\n  Early stopping at epoch {epoch + 1}")
            stopped_early = True
            break

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_loss': best_val_loss,
        'patience_counter': patience_counter,
    }, str(ckpt_path))

if ckpt_path.exists():
    ckpt_path.unlink()

elapsed = (datetime.now() - t0).total_seconds() / 60
print(f"\n  Done: {elapsed:.1f} min  |  Best val_loss: {best_val_loss:.4f}"
      f"{' (early stop)' if stopped_early else ''}")
print(f"  Model saved: {out_dir}")
